# FMAT3888 — Question 3: Strategic Asset Allocation

This notebook calculates the long-only minimum-variance portfolio for 30%, 70%, and 90% Growth exposure targets. It is designed to reuse the group's return sample and constraints from Questions 1–2. The current local repository does not contain those inputs, so the notebook reports missing data instead of inventing results.

For each target, the optimization minimizes estimated portfolio variance subject to weights summing to 1, APRA-weighted Growth exposure matching the target, and any per-asset maximum agreed by the group.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
from q3_saa import ANNUALIZATION, read_returns, solve_saa_profiles
print('Q3 calculation module loaded. Confirm the inputs against Q1/Q2 before running.')

## 1. Confirm the shared inputs

Set the paths and assumptions only after the group confirms the Q1/Q2 return file, asset names, frequency, annual risk-free rate, and any weight cap. Returns must be periodic simple returns in decimal form (`0.02` means 2%). The classification mapping needs one row for every return column. Keep raw Bloomberg data out of this public repository.

In [ ]:
REPO_ROOT = NOTEBOOK_DIR.parent
RETURNS_FILE = REPO_ROOT / 'data/processed/returns.csv'
CLASSIFICATION_FILE = NOTEBOOK_DIR / 'asset_growth_mapping.csv'
FREQUENCY = None  # choose only after confirming the Q1/Q2 frequency
RISK_FREE_RATE_PCT = None  # replace with the agreed annual rate, e.g. 4.0 for 4%
MAX_WEIGHT = None  # set the Q2 per-asset cap if there is one
if FREQUENCY not in ANNUALIZATION:
    print(f'Choose the confirmed Q1/Q2 frequency from: {list(ANNUALIZATION)}')
print('Returns file exists:', RETURNS_FILE.exists())
print('Classification file exists:', CLASSIFICATION_FILE.exists())
if RISK_FREE_RATE_PCT is None:
    print('Set RISK_FREE_RATE_PCT after confirming it with the group.')

## 2. Estimate and compare the three portfolios

The script annualizes the sample mean and covariance using the selected frequency. The Sharpe ratio is `(annual expected return − annual risk-free rate) / annual volatility`. Confirm all choices with Q1/Q2 before interpreting the outputs.

In [ ]:
if FREQUENCY not in ANNUALIZATION:
    print('Waiting for the group to confirm return frequency.')
elif not RETURNS_FILE.exists() or not CLASSIFICATION_FILE.exists():
    print('Waiting for the Q1/Q2 returns and completed asset mapping. No results have been calculated.')
elif RISK_FREE_RATE_PCT is None:
    print('Set the agreed annual risk-free rate above, then run this cell again.')
else:
    returns = read_returns(RETURNS_FILE)
    classification = pd.read_csv(CLASSIFICATION_FILE)
    q3_metrics, q3_weights = solve_saa_profiles(
        returns, classification,
        periods_per_year=ANNUALIZATION[FREQUENCY],
        risk_free_rate=RISK_FREE_RATE_PCT / 100.0,
        max_weight=MAX_WEIGHT,
    )
    display(q3_metrics.style.format({
        'target_growth': '{:.0%}', 'actual_growth': '{:.1%}',
        'defensive_share': '{:.1%}', 'expected_return_annual': '{:.2%}',
        'volatility_annual': '{:.2%}', 'risk_free_rate_annual': '{:.2%}',
        'sharpe_ratio': '{:.3f}', 'variance_annual': '{:.6f}'
    }))

In [ ]:
if 'q3_weights' in globals():
    display(q3_weights.pivot(index='asset', columns='target_growth', values='weight').style.format('{:.1%}'))
    OUTPUT_DIR = REPO_ROOT / 'outputs' / 'q3'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    q3_metrics.to_csv(OUTPUT_DIR / 'q3_portfolio_metrics.csv', index=False)
    q3_weights.to_csv(OUTPUT_DIR / 'q3_portfolio_weights.csv', index=False)
    print('Saved the metrics and weights under outputs/.')
else:
    print('Run the calculation cell first; shared inputs are not available yet.')

## 3. Interpretation for the report and presentation

Once real inputs are connected, compare the three profiles using expected return, volatility, and Sharpe ratio. Explain which asset classes receive the largest weights, how covariance affects diversification, and what risk/return trade-off each profile represents. Do not assume the 90% Growth profile must have the highest estimated return or Sharpe ratio. State that the estimates depend on the sample, classifications, frequency, risk-free rate, and constraints. Use `../reports/Q3_section_template.md` as the report-writing structure.